# From Personas to Talks — Colab Setup

**Paper:** Wu et al., EMNLP 2025  
**Steps below:** install deps → mount Drive → set API keys → copy project → smoke-test RQ1

> **Before running:** add your API keys as Colab secrets (🔑 icon in the left sidebar):  
> - `GEMINI_API_KEY` — required (primary LLM)  
> - `OPENAI_API_KEY` — optional (fallback LLM)

## Step 1 — Install dependencies

In [1]:
%%capture install_log
# Install all packages pinned in requirements.txt.
# %%capture suppresses the wall of pip output; remove it if you need to debug.
%pip install \
    "google-genai>=1.0.0" \
    "datasets>=2.19.0" \
    "numpy>=1.26.0" \
    "pyyaml>=6.0.1" \
    "openai>=1.0.0" \
    "scipy>=1.13.0"

print("Installation complete.")

## Step 2 — Mount Google Drive

The project files live in your Drive. Adjust `DRIVE_PROJECT_PATH` below to match where you uploaded the `personas_to_talks/` folder.

In [2]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# ── Edit this path to match your Drive layout ─────────────────────────────────
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/personas_to_talks"
# ─────────────────────────────────────────────────────────────────────────────

import os
if not os.path.isdir(DRIVE_PROJECT_PATH):
    raise FileNotFoundError(
        f"Project not found at '{DRIVE_PROJECT_PATH}'.\n"
        "Upload the personas_to_talks/ folder to your Drive and update DRIVE_PROJECT_PATH."
    )
print(f"Drive mounted. Project found at: {DRIVE_PROJECT_PATH}")

Mounted at /content/drive
Drive mounted. Project found at: /content/drive/MyDrive/personas_to_talks


## Step 3 — Load API keys from Colab secrets

Keys are read from Colab's secret store (🔑 sidebar → **Add new secret**).  
They are never written to disk or printed.

In [3]:
import os
from google.colab import userdata

try:
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("GEMINI_API_KEY loaded.")
except Exception:
    print("WARNING: GEMINI_API_KEY not found.")

try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("OPENAI_API_KEY loaded.")
except Exception:
    print("INFO: OPENAI_API_KEY not set.")

if not os.environ.get("GEMINI_API_KEY") and not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("No API keys available. Set at least GEMINI_API_KEY in Colab secrets.")

GEMINI_API_KEY loaded.
OPENAI_API_KEY loaded.


## Step 4 — Copy project to /content/ and set working directory

We copy from Drive to `/content/personas_to_talks/` so file I/O is fast (Drive FUSE is slow for many small reads/writes). Intermediate outputs written during a run are saved back to the copy — re-run the copy cell to sync changes back to Drive if needed.

In [4]:
import shutil

LOCAL_PROJECT = "/content/personas_to_talks"

if os.path.isdir(LOCAL_PROJECT):
    shutil.rmtree(LOCAL_PROJECT)

shutil.copytree(DRIVE_PROJECT_PATH, LOCAL_PROJECT)
os.chdir(LOCAL_PROJECT)
print(f"Working directory: {os.getcwd()}")
print("Project files:")
for entry in sorted(os.listdir(LOCAL_PROJECT)):
    print(f"  {entry}")

Working directory: /content/personas_to_talks
Project files:
  .DS_Store
  colab_setup.ipynb
  config.yaml
  data
  main.py
  main_phase2.py
  outputs
  reproduce.sh
  requirements.txt
  src


## Step 4b — Configure LLM provider

In [5]:
import yaml

config_path = os.path.join(LOCAL_PROJECT, "config.yaml")
with open(config_path) as f:
    cfg = yaml.safe_load(f)

cfg["llm"]["primary_provider"] = "gemini"
cfg["llm"]["fallback_provider"] = "openai"
cfg["llm"]["gemini_model"] = "gemini-2.5-flash"
cfg["llm"]["openai_model"] = "gpt-4o-mini"
cfg["rq1"]["skip_persona_filter"] = True

with open(config_path, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)

print("Config: gemini-2.5-flash primary, gpt-4o-mini fallback, filter skipped.")

Config: gemini-2.5-flash primary, gpt-4o-mini fallback, filter skipped.


### (Optional) Sync outputs back to Drive

Run this cell any time you want to persist intermediate outputs to Drive so they survive a runtime restart.

In [7]:
# Run this cell to copy outputs/ back to Drive.
LOCAL_OUTPUTS = os.path.join(LOCAL_PROJECT, "outputs")
DRIVE_OUTPUTS = os.path.join(DRIVE_PROJECT_PATH, "outputs")

if os.path.isdir(LOCAL_OUTPUTS):
    if os.path.isdir(DRIVE_OUTPUTS):
        shutil.rmtree(DRIVE_OUTPUTS)
    shutil.copytree(LOCAL_OUTPUTS, DRIVE_OUTPUTS)
    print(f"Outputs synced to Drive: {DRIVE_OUTPUTS}")
else:
    print("No outputs/ directory yet — run a pipeline first.")

Outputs synced to Drive: /content/drive/MyDrive/personas_to_talks/outputs


## Step 5 — Smoke test: RQ1 (ESConv + CAMS + Dreaddit)

Runs the full RQ1 pipeline with the caps set in `config.yaml` (`max_dialogues_per_dataset: 200`).  
Lower the cap before running if you want a quick sanity check:

In [8]:
# Optional: lower the per-dataset cap for a fast smoke test (e.g. 5 dialogues).
# Set to None to use the full dataset as configured in config.yaml.
SMOKE_TEST_CAP = 5

if SMOKE_TEST_CAP is not None:
    import yaml
    config_path = os.path.join(LOCAL_PROJECT, "config.yaml")
    with open(config_path) as f:
        cfg = yaml.safe_load(f)
    cfg["rq1"]["max_dialogues_per_dataset"] = SMOKE_TEST_CAP
    with open(config_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)
    print(f"Smoke-test cap set to {SMOKE_TEST_CAP} dialogues per dataset.")
else:
    print("Using cap from config.yaml.")

Smoke-test cap set to 5 dialogues per dataset.


In [ ]:
# Run RQ1. Output is streamed live.
!python main.py --rq 1

## (Optional) Run RQ2 or RQ3

In [ ]:
# !python main.py --rq 2

In [ ]:
# !python main.py --rq 3

In [ ]:
# !python main.py --rq all

## Phase 2 — Global Emotional Profile (Ross)

In [9]:
# Run Phase 2: build Ross's Global Emotional Profile and evaluate emotion prediction
!python main_phase2.py --character Ross

23:19:42 [WARNING] src.llm_client: ANTHROPIC_API_KEY not set; Anthropic provider unavailable.
23:19:43 [INFO] __main__: === Step 1: Loading MELD 'train' split for 'Ross' ===
23:19:43 [INFO] src.meld_loader: Loading MELD from local CSV: /content/personas_to_talks/data/train_sent_emo.csv
23:19:43 [INFO] src.meld_loader: Loaded 9989 rows from train split.
23:19:43 [INFO] src.meld_loader: Found 374 conversations for 'Ross'.
23:19:43 [INFO] __main__: Loaded 374 train conversations.
23:19:43 [INFO] __main__: === Step 2: Aggregating Global Emotional Profile ===
23:19:43 [INFO] src.profile_aggregator: Aggregating batch 1/19 for 'Ross' (20 convs).
23:19:43 [INFO] google_genai.models: AFC is enabled with max remote calls: 10.
23:20:00 [INFO] httpx: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
23:20:00 [INFO] google_genai.models: AFC is enabled with max remote calls: 10.
23:20:04 [INFO] httpx: HTTP Request: POST http